# Geothermal Plants, Māori Land Property Titles & Iwi Rohe — Data Processing & Interactive Map Builder (EN & JA)

This notebook processes the GIS and tabular datasets from `plants_data`, `property_data`, and `rohe_data` and builds standalone interactive web map applications in both **English (`map_app/index.html`)** and **Japanese (`map_app/index_ja.html`)**.

### Data Sources Handled
1. **Geothermal Power Stations (`../plants_data/geothermal_plants.gpkg`)**
   - Point locations, operator details, generation technology, and installed capacity (MW) in the Taupō Volcanic Zone.
2. **Māori Land Consolidated Data + Property Titles (`../property_data/20250630_maori_land_consolidated.csv` + `../property_data/nz-property-titles.shp`)**
   - Joins Māori Land Court (MLC) consolidated title records (`LINZ Title Reference`) with LINZ property title boundaries (`title_no`).
   - Distinguishes **Standard Māori Land (`is_aggregated == 0`)** from **Aggregated Ownership (`is_aggregated == 1`, owners shared across multiple blocks)**.
   - Uses `is_rep == 1` to place representative owner-count labels (`No. of Owners`) once per ownership group (`label_group`).
3. **Iwi Areas of Interest / Rohe (`../rohe_data/Iwi_AreasOfInterest.shp`)**
   - Filters Te Puni Kōkiri Iwi Areas of Interest to the Central North Island / Taupō–Reporoa geothermal region.
   - Highlights **Ngāti Tahu / Ngāti Whaoa** and **Ngāti Tūwharetoa** (plus their overlapping area of interest) by default, while keeping neighbouring regional iwi toggleable.

---
### Base Maps Configured
- **Esri World Topo Map** (Default base map layer)
- **Google Maps (Roadmap)**
- **Google Maps (Hybrid Satellite)**
- **Google Maps (Terrain)**

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, mapping

# ==============================================================================
# 1. FILE PATHS (Relative to `map_app/` directory)
# ==============================================================================
BASE_DATA_DIR = Path("..")
PLANTS_GPKG_PATH = BASE_DATA_DIR / "plants_data" / "geothermal_plants.gpkg"
# Tip: To use the root-level updated_plants.gpkg instead, uncomment the line below:
# PLANTS_GPKG_PATH = BASE_DATA_DIR / "updated_plants.gpkg"

PROPERTY_SHP_PATH = BASE_DATA_DIR / "property_data" / "nz-property-titles.shp"
MAORI_CSV_PATH = BASE_DATA_DIR / "property_data" / "20250630_maori_land_consolidated.csv"
ROHE_SHP_PATH = BASE_DATA_DIR / "rohe_data" / "Iwi_AreasOfInterest.shp"

PROCESSED_DIR = Path("processed_data")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_HTML_EN_PATH = Path("index.html")
OUTPUT_HTML_JA_PATH = Path("index_ja.html")

# ==============================================================================
# 2. BASE MAP PROVIDERS (Esri World Topo Default + 3 Google Maps Layers)
# ==============================================================================
ESRI_WORLD_TOPO_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}"
GOOGLE_MAPS_ROADMAP_URL = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}"
GOOGLE_MAPS_HYBRID_URL = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}"
GOOGLE_MAPS_TERRAIN_URL = "https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}"

# ==============================================================================
# 3. MAP VIEW, OWNER VISIBILITY & GEOMETRY PARAMETERS
# ==============================================================================
DEFAULT_CENTER = [-38.55, 176.18]  # [Latitude, Longitude] centered on Taupō–Reporoa geothermal zone
DEFAULT_ZOOM = 10

# Geometry simplification tolerances (in meters, EPSG:2193) to keep browser rendering fast
PARCEL_SIMPLIFY_TOLERANCE_M = 1.5
ROHE_SIMPLIFY_TOLERANCE_M = 15.0
COORD_PRECISION = 5  # ~1 meter precision in WGS84 (EPSG:4326)

# Owner Count Labels visibility on startup (False = hidden by default so map is clean; toggleable in UI)
SHOW_OWNER_LABELS_BY_DEFAULT = False
DEFAULT_MIN_OWNERS_FOR_LABELS = 0  # 0 to max_owners (all)

# ==============================================================================
# 4. IWI ROHE SELECTION & STYLING
# ==============================================================================
# Primary Rohe shown by default on map load
PRIMARY_ROHE_STYLES = {
    "Ngāti Tahu / Ngāti Whaoa": {
        "color": "#be123c",       # Crimson border
        "fillColor": "#f43f5e",   # Rose fill
        "fillOpacity": 0.14,
        "weight": 2.5,
        "dashArray": "6, 4",
        "default_on": True,
    },
    "Ngāti Tūwharetoa": {
        "color": "#0369a1",       # Deep sky-blue border
        "fillColor": "#0ea5e9",   # Sky-blue fill
        "fillOpacity": 0.14,
        "weight": 2.5,
        "dashArray": "8, 4, 2, 4",
        "default_on": True,
    },
}

# Other Central North Island / Geothermal Region Iwi (available as toggleable layers)
SECONDARY_ROHE_PALETTE = [
    ("Ngāti Rangitihi", "#7c3aed"),
    ("Tūhourangi", "#0d9488"),
    ("Raukawa", "#d97706"),
    ("Pouākani (Land Block)", "#4f46e5"),
    ("Tūhoe", "#15803d"),
    ("Ngāti Whakaue", "#b45309"),
    ("Ngāti Mākino", "#9333ea"),
    ("Ngāti Kea / Ngāti Tuarā", "#0891b2"),
    ("Ngāti Rongomai", "#c026d3"),
    ("Ngāti Tarāwhai", "#2563eb"),
    ("Ngāti Rangiwewehi", "#059669"),
    ("Ngāti Pikiao", "#dc2626"),
    ("Ngāti Manawa", "#65a30d"),
    ("Ngāti Whare", "#475569"),
]

# ==============================================================================
# 5. MĀORI LAND & GEOTHERMAL PLANT STYLING
# ==============================================================================
STYLE_STANDARD_MAORI = {
    "color": "#1e293b",       # Slate-800 border
    "fillColor": "#64748b",   # Slate-500 fill (matches report figures)
    "fillOpacity": 0.55,
    "weight": 1.0,
}

STYLE_AGGREGATED_MAORI = {
    "color": "#92400e",       # Amber-800 border
    "fillColor": "#f59e0b",   # Amber-500 / Orange fill (matches report figures)
    "fillOpacity": 0.72,
    "weight": 1.4,
}

PLANT_MARKER_COLOR = "#dc2626"  # Red triangle marker

print("Configuration loaded successfully.")
print("Default base layer: Esri World Topo Map (+ 3 Google Maps layers)")


Configuration loaded successfully.
Default base layer: Esri World Topo Map (+ 3 Google Maps layers)


## Step 1: Load & Process Geothermal Power Stations (`plants_data/geothermal_plants.gpkg`)

We load the geothermal power station locations from `geothermal_plants.gpkg` (in `EPSG:2193` / NZTM2000), convert their coordinates to `EPSG:4326` (WGS84 latitude/longitude) for web mapping, and export a clean GeoJSON file.

In [2]:
# Load geothermal plants GeoPackage
plants_gdf = gpd.read_file(PLANTS_GPKG_PATH)
print(f"Loaded {len(plants_gdf)} geothermal power stations (CRS: {plants_gdf.crs})")

# Reproject to WGS84 (EPSG:4326) for Leaflet web mapping
plants_wgs84 = plants_gdf.to_crs(epsg=4326).copy()
plants_wgs84["lon"] = plants_wgs84.geometry.x.round(6)
plants_wgs84["lat"] = plants_wgs84.geometry.y.round(6)

# Save processed GeoJSON
plants_geojson_path = PROCESSED_DIR / "geothermal_plants.geojson"
plants_wgs84.to_file(plants_geojson_path, driver="GeoJSON")
print(f"Saved processed plants GeoJSON to: {plants_geojson_path}")

# Display table of geothermal plants
plants_wgs84[["name", "operator", "capacity_mw", "type", "easting", "northing", "lat", "lon"]]


Loaded 6 geothermal power stations (CRS: EPSG:2193)
Saved processed plants GeoJSON to: processed_data\geothermal_plants.geojson


,name,operator,capacity_mw,type,easting,northing,lat,lon
0,Ohaaki,Contact Energy,44,Flash,1887145,5730689,-38.525896,176.293797
1,Ngatamariki,Mercury NZ,132,Binary,1878510,5728660,-38.546896,176.195700
2,Tauhara,Contact Energy,174,Triple Flash,1874682,5715567,-38.665901,176.157001
3,Nga Awa Purua,Mercury NZ / Tauhara North No.2 Trust,140,Triple Flash,1877209,5721337,-38.613196,176.183703
4,Wairakei,Contact Energy,132,Flash / Binary,1870195,5720210,-38.625500,176.103697
5,Mokai,Tuaropaki Trust / Mercury NZ,113,Combined Cycle (Steam + Binary),1855082,5731196,-38.531100,175.926300


## Step 2: Load & Process Iwi Areas of Interest / Rohe (`rohe_data/Iwi_AreasOfInterest.shp`)

We filter `Iwi_AreasOfInterest.shp` to the Central North Island / Taupō–Reporoa geothermal region iwi configured above. We also compute the exact geometric intersection between **Ngāti Tahu / Ngāti Whaoa** and **Ngāti Tūwharetoa** so the overlapping area of interest can be toggled directly on the map.

In [3]:
# Load Iwi Areas of Interest shapefile (EPSG:2193)
rohe_gdf = gpd.read_file(ROHE_SHP_PATH)
print(f"Total Iwi Rohe in dataset: {len(rohe_gdf)} (CRS: {rohe_gdf.crs})")

# Combine primary and secondary iwi lists
target_iwi_names = list(PRIMARY_ROHE_STYLES.keys()) + [name for name, _ in SECONDARY_ROHE_PALETTE]
rohe_regional = rohe_gdf[rohe_gdf["Name"].isin(target_iwi_names)].copy()

# Calculate area in km2 in NZTM (EPSG:2193) before simplifying
rohe_regional["area_km2"] = (rohe_regional.geometry.area / 1e6).round(1)
rohe_regional["is_primary"] = rohe_regional["Name"].isin(PRIMARY_ROHE_STYLES.keys())

# Compute the overlapping area between Ngāti Tahu / Ngāti Whaoa and Ngāti Tūwharetoa
tahu_geom = rohe_regional.loc[rohe_regional["Name"] == "Ngāti Tahu / Ngāti Whaoa", "geometry"].unary_union
tuwharetoa_geom = rohe_regional.loc[rohe_regional["Name"] == "Ngāti Tūwharetoa", "geometry"].unary_union
overlap_geom_nztm = tahu_geom.intersection(tuwharetoa_geom)
overlap_area_km2 = round(overlap_geom_nztm.area / 1e6, 1)
print(f"Ngāti Tahu / Ngāti Whaoa ∩ Ngāti Tūwharetoa overlap area: {overlap_area_km2:,.1f} km²")

# Simplify geometries slightly in NZTM (meters) and reproject to WGS84 (EPSG:4326)
rohe_regional["geometry"] = rohe_regional.geometry.simplify(ROHE_SIMPLIFY_TOLERANCE_M, preserve_topology=True)
rohe_wgs84 = rohe_regional.to_crs(epsg=4326).copy()

overlap_gdf_wgs84 = gpd.GeoDataFrame(
    [{
        "Name": "Overlapping Area (Ngāti Tahu / Ngāti Whaoa ∩ Ngāti Tūwharetoa)",
        "Source": "Derived geometric intersection from TPK Iwi Areas of Interest",
        "TPK_Code": "OVL",
        "area_km2": overlap_area_km2,
        "is_primary": True,
    }],
    geometry=[overlap_geom_nztm.simplify(ROHE_SIMPLIFY_TOLERANCE_M, preserve_topology=True)],
    crs=rohe_gdf.crs,
).to_crs(epsg=4326)

# Save processed Rohe GeoJSON
rohe_geojson_path = PROCESSED_DIR / "rohe_central_ni.geojson"
rohe_wgs84.to_file(rohe_geojson_path, driver="GeoJSON")
print(f"Saved {len(rohe_wgs84)} regional Rohe polygons to: {rohe_geojson_path}")

rohe_wgs84[["Name", "TPK_Code", "area_km2", "is_primary", "Source"]].sort_values(
    by=["is_primary", "Name"], ascending=[False, True]
).reset_index(drop=True)


Total Iwi Rohe in dataset: 112 (CRS: EPSG:2193)
Ngāti Tahu / Ngāti Whaoa ∩ Ngāti Tūwharetoa overlap area: 1,583.3 km²
Saved 16 regional Rohe polygons to: processed_data\rohe_central_ni.geojson


,Name,TPK_Code,area_km2,is_primary,Source
0,Ngāti Tahu / Ngāti Whaoa,41,1942.1,True,TPK
1,Ngāti Tūwharetoa,42,11335.2,True,TPK
2,Ngāti Kea / Ngāti Tuarā,90,307.9,False,TPK
3,Ngāti Manawa,49,2519.0,False,TPK
4,Ngāti Mākino,30,917.8,False,TPK
5,Ngāti Pikiao,31,1002.1,False,TPK
6,Ngāti Rangitihi,33,3650.4,False,TPK
7,Ngāti Rangiwewehi,34,974.2,False,TPK
8,Ngāti Rongomai,92,44.3,False,TPK
9,Ngāti Tarāwhai,36,25.4,False,TPK


## Step 3: Combine `20250630_maori_land_consolidated.csv` with `nz-property-titles.shp`

Here we join the consolidated Māori land dataset (`20250630_maori_land_consolidated.csv`) with the LINZ property title polygons (`nz-property-titles.shp`):
- **Join Key**: `nz-property-titles.shp` [`title_no`] $\leftrightarrow$ `20250630_maori_land_consolidated.csv` [`LINZ Title Reference`].
- **Ownership Categories**:
  - `is_aggregated == 0`: Standard Māori land title (owners specific to that block).
  - `is_aggregated == 1`: Aggregated ownership (multiple blocks share a single consolidated owner list across `agg_group` / `label_group`).
- **Representative Points for Owner Count Labels (`is_rep == 1`)**:
  - Following the Māori Land Court consolidated methodology, `is_rep == 1` marks one representative parcel per ownership group (`label_group`) so that owner counts (`No. of Owners`) are displayed once per group without duplicate overlapping labels.

In [4]:
# 1. Read consolidated Māori Land CSV
maori_csv = pd.read_csv(MAORI_CSV_PATH)
print(f"Loaded Māori Land Consolidated CSV: {len(maori_csv):,} rows")

# 2. Read LINZ Property Titles shapefile
titles_gdf = gpd.read_file(PROPERTY_SHP_PATH)
print(f"Loaded LINZ Property Titles Shapefile: {len(titles_gdf):,} polygons (CRS: {titles_gdf.crs})")

# 3. Inner join on title_no == LINZ Title Reference
maori_titles_gdf = titles_gdf.merge(
    maori_csv,
    left_on="title_no",
    right_on="LINZ Title Reference",
    how="inner",
)
print(f"Matched Māori Land property titles in study extent: {len(maori_titles_gdf):,} polygons")

# 4. Compute area in hectares (ha) in EPSG:2193 (NZTM meters)
maori_titles_gdf["area_ha"] = (maori_titles_gdf.geometry.area / 10000.0).round(2)

# Clean/normalize columns for display and filtering
maori_titles_gdf["is_aggregated"] = maori_titles_gdf["is_aggregated"].fillna(0).astype(int)
maori_titles_gdf["is_imputed"] = maori_titles_gdf["is_imputed"].fillna(0).astype(int)
maori_titles_gdf["is_rep"] = maori_titles_gdf["is_rep"].fillna(0).astype(int)
maori_titles_gdf["ownership_category"] = np.where(
    maori_titles_gdf["is_aggregated"] == 1,
    "Aggregated ownership (shared across blocks)",
    "Māori land (standard)",
)

# 5. Extract representative points (inside each polygon) for owner-count labels (is_rep == 1)
rep_points_nztm = maori_titles_gdf.geometry.representative_point()
rep_points_wgs84 = gpd.GeoSeries(rep_points_nztm, crs=maori_titles_gdf.crs).to_crs(epsg=4326)
maori_titles_gdf["rep_lat"] = rep_points_wgs84.y.round(5)
maori_titles_gdf["rep_lon"] = rep_points_wgs84.x.round(5)

# Determine the exact maximum number of owners across all matched parcels
MAX_OWNERS_IN_DATA = int(maori_titles_gdf["No. of Owners"].max())
print(f"Owner count range in matched data: 0 to {MAX_OWNERS_IN_DATA:,} owners")

# 6. Save full-resolution matched GeoPackage in NZTM (EPSG:2193) for GIS software (e.g., QGIS)
maori_gpkg_path = PROCESSED_DIR / "maori_land_matched.gpkg"
maori_titles_gdf.to_file(maori_gpkg_path, driver="GPKG", layer="maori_land_matched")
print(f"Saved full-resolution GeoPackage to: {maori_gpkg_path}")

# 7. Simplify geometries slightly (1.5m tolerance) and reproject to EPSG:4326 for fast web rendering
maori_wgs84 = maori_titles_gdf.copy()
maori_wgs84["geometry"] = maori_wgs84.geometry.simplify(PARCEL_SIMPLIFY_TOLERANCE_M, preserve_topology=True)
maori_wgs84 = maori_wgs84.to_crs(epsg=4326)

# Summary breakdown by ownership type
summary_df = (
    maori_wgs84.groupby("ownership_category", as_index=False)
    .agg(
        parcel_count=("title_no", "count"),
        unique_label_groups=("label_group", "nunique"),
        representative_labels=("is_rep", "sum"),
        total_area_ha=("area_ha", lambda s: round(s.sum(), 1)),
        median_owners=("No. of Owners", "median"),
        max_owners=("No. of Owners", "max"),
    )
)
print("\n=== Summary of Matched Māori Land Titles ===")
print(summary_df.to_string(index=False))

maori_wgs84[[
    "title_no", "ownership_category", "No. of Owners", "number_own",
    "area_ha", "type", "land_distr", "group_id", "agg_group", "label_group", "is_rep"
]].head(10)


Loaded Māori Land Consolidated CSV: 27,998 rows
Loaded LINZ Property Titles Shapefile: 119,626 polygons (CRS: EPSG:2193)
Matched Māori Land property titles in study extent: 4,757 polygons
Owner count range in matched data: 0 to 17,737 owners
Saved full-resolution GeoPackage to: processed_data\maori_land_matched.gpkg

=== Summary of Matched Māori Land Titles ===
                         ownership_category  parcel_count  unique_label_groups  representative_labels  total_area_ha  median_owners  max_owners
Aggregated ownership (shared across blocks)            94                   23                     22         6417.5          510.0      6935.0
                      Māori land (standard)          4663                 4486                   4484       377263.4           52.0     17737.0


,title_no,ownership_category,No. of Owners,number_own,area_ha,type,land_distr,group_id,agg_group,label_group,is_rep
0,478365,Māori land (standard),77.0,106,0.61,Freehold,Taranaki,s1_3888,NaN,s1_3888,1
1,475099,Māori land (standard),6.0,6,16.21,Freehold,South Auckland,s1_26381,NaN,s1_26381,1
2,480456,Māori land (standard),22.0,23,0.14,Freehold,Taranaki,s1_3853,NaN,s1_3853,1
3,276068,Māori land (standard),62.0,87,0.28,Freehold,South Auckland,s1_424,NaN,s1_424,1
4,408711,Māori land (standard),79.0,5,154.93,Freehold,South Auckland,s1_3486,NaN,s1_3486,1
5,448773,Māori land (standard),88.0,118,0.00,Freehold,Wellington,s1_2384,NaN,s1_2384,1
6,464195,Māori land (standard),395.0,9,18.95,Freehold,Wellington,s1_3764,NaN,s1_3764,1
7,342347,Māori land (standard),1414.0,9,92.39,Freehold,Wellington,s1_3763,NaN,s1_3763,1
8,SA11B/1233,Māori land (standard),NaN,1,377.53,Freehold,South Auckland,s1_2382,NaN,s1_2382,1
9,423025,Māori land (standard),58.0,61,0.08,Freehold,South Auckland,s1_21388,NaN,s1_21388,1


## Step 4: Build English (`index.html`) & Japanese (`index_ja.html`) Standalone Interactive Map Apps

We now generate both like-for-like standalone web applications:
- **`index.html`**: English edition (with a header link to switch to the Japanese edition).
- **`index_ja.html`**: Japanese edition (with a header link to switch to the English edition).

### Base Maps Included:
1. **Esri World Topo Map** (Default base map layer on startup)
2. **Google Maps (Roadmap)**
3. **Google Maps (Hybrid Satellite)**
4. **Google Maps (Terrain)**

In [5]:
def round_coords(coords, precision=COORD_PRECISION):
    """Recursively round GeoJSON coordinate arrays to keep the standalone HTML compact and fast."""
    if isinstance(coords, (list, tuple)):
        if len(coords) > 0 and isinstance(coords[0], (int, float)):
            return [round(float(c), precision) for c in coords]
        return [round_coords(c, precision) for c in coords]
    return coords

# 1. Serialize compact GeoJSON for Māori Land parcels
maori_features = []
for _, row in maori_wgs84.iterrows():
    geom_dict = mapping(row.geometry)
    geom_dict["coordinates"] = round_coords(geom_dict["coordinates"], COORD_PRECISION)
    owners_val = None if pd.isna(row["No. of Owners"]) else int(round(float(row["No. of Owners"])))
    linz_owners_val = None if pd.isna(row["number_own"]) else int(row["number_own"])
    maori_features.append({
        "type": "Feature",
        "geometry": geom_dict,
        "properties": {
            "id": int(row["id"]),
            "title_no": str(row["title_no"] or ""),
            "type": str(row["type"] or ""),
            "status": str(row["status"] or ""),
            "land_distr": str(row["land_distr"] or ""),
            "issue_date": str(row["issue_date"] or "")[:10],
            "estate_des": str(row["estate_des"] or "").replace("\r\n", " | ").replace("\n", " | "),
            "number_own": linz_owners_val,
            "mlc_owners": owners_val,
            "area_ha": float(row["area_ha"]),
            "group_id": "" if pd.isna(row["group_id"]) else str(row["group_id"]),
            "agg_group": "" if pd.isna(row["agg_group"]) else str(row["agg_group"]),
            "label_group": "" if pd.isna(row["label_group"]) else str(row["label_group"]),
            "is_imputed": int(row["is_imputed"]),
            "is_aggregated": int(row["is_aggregated"]),
            "is_rep": int(row["is_rep"]),
            "rep_lat": float(row["rep_lat"]),
            "rep_lon": float(row["rep_lon"]),
        },
    })

maori_geojson_obj = {"type": "FeatureCollection", "features": maori_features}

# Save the web-optimized GeoJSON to processed_data/
with open(PROCESSED_DIR / "maori_land_matched.geojson", "w", encoding="utf-8") as f:
    json.dump(maori_geojson_obj, f, ensure_ascii=False)

# 2. Serialize Rohe features (Primary + Overlap + Secondary)
rohe_layers_data = []

for iwi_name, style_cfg in PRIMARY_ROHE_STYLES.items():
    sub = rohe_wgs84[rohe_wgs84["Name"] == iwi_name]
    if len(sub) > 0:
        row = sub.iloc[0]
        geom_dict = mapping(row.geometry)
        geom_dict["coordinates"] = round_coords(geom_dict["coordinates"], COORD_PRECISION)
        rohe_layers_data.append({
            "name": iwi_name,
            "name_ja": iwi_name,
            "category": "primary",
            "default_on": style_cfg["default_on"],
            "style": style_cfg,
            "tpk_code": str(row["TPK_Code"] or ""),
            "area_km2": float(row["area_km2"]),
            "source": str(row["Source"] or ""),
            "geometry": geom_dict,
        })

# Add Overlap layer
ovl_row = overlap_gdf_wgs84.iloc[0]
ovl_geom = mapping(ovl_row.geometry)
ovl_geom["coordinates"] = round_coords(ovl_geom["coordinates"], COORD_PRECISION)
rohe_layers_data.append({
    "name": "Overlap: Ngāti Tahu / Ngāti Whaoa ∩ Ngāti Tūwharetoa",
    "name_ja": "重複領域: Ngāti Tahu / Ngāti Whaoa ∩ Ngāti Tūwharetoa",
    "category": "overlap",
    "default_on": False,
    "style": {
        "color": "#4c1d95",
        "fillColor": "#6d28d9",
        "fillOpacity": 0.22,
        "weight": 2.0,
        "dashArray": "3, 5",
    },
    "tpk_code": "OVL",
    "area_km2": float(ovl_row["area_km2"]),
    "source": str(ovl_row["Source"]),
    "geometry": ovl_geom,
})

for iwi_name, hex_color in SECONDARY_ROHE_PALETTE:
    sub = rohe_wgs84[rohe_wgs84["Name"] == iwi_name]
    if len(sub) > 0:
        row = sub.iloc[0]
        geom_dict = mapping(row.geometry)
        geom_dict["coordinates"] = round_coords(geom_dict["coordinates"], COORD_PRECISION)
        rohe_layers_data.append({
            "name": iwi_name,
            "name_ja": iwi_name.replace("(Land Block)", "（土地ブロック）"),
            "category": "secondary",
            "default_on": False,
            "style": {
                "color": hex_color,
                "fillColor": hex_color,
                "fillOpacity": 0.12,
                "weight": 2.0,
                "dashArray": "5, 5",
            },
            "tpk_code": str(row["TPK_Code"] or ""),
            "area_km2": float(row["area_km2"]),
            "source": str(row["Source"] or ""),
            "geometry": geom_dict,
        })

# 3. Serialize Geothermal Plants
plants_list = []
for _, row in plants_wgs84.iterrows():
    plants_list.append({
        "name": str(row["name"]),
        "operator": str(row["operator"]),
        "capacity_mw": int(row["capacity_mw"]),
        "type": str(row["type"]),
        "easting": int(row["easting"]),
        "northing": int(row["northing"]),
        "lat": float(row["lat"]),
        "lon": float(row["lon"]),
    })

app_config = {
    "center": DEFAULT_CENTER,
    "zoom": DEFAULT_ZOOM,
    "maxOwners": MAX_OWNERS_IN_DATA,
    "showOwnerLabelsByDefault": SHOW_OWNER_LABELS_BY_DEFAULT,
    "defaultMinOwnersLabel": DEFAULT_MIN_OWNERS_FOR_LABELS,
    "styleStandardMaori": STYLE_STANDARD_MAORI,
    "styleAggregatedMaori": STYLE_AGGREGATED_MAORI,
    "plantMarkerColor": PLANT_MARKER_COLOR,
    "tileUrls": {
        "esriTopo": ESRI_WORLD_TOPO_URL,
        "googleRoadmap": GOOGLE_MAPS_ROADMAP_URL,
        "googleHybrid": GOOGLE_MAPS_HYBRID_URL,
        "googleTerrain": GOOGLE_MAPS_TERRAIN_URL,
    },
}

UI_TEXT = {
    "en": {
        "html_lang": "en",
        "page_title": "Taupō–Reporoa Geothermal & Māori Land Interactive GIS Map",
        "header_title": "Geothermal & Māori Land GIS Explorer",
        "header_sub": "Taupō–Reporoa Geothermal Zone • LINZ Titles + Māori Land Court (2025) & TPK Iwi Rohe",
        "lang_switch_label": "🇯🇵 日本語版へ",
        "lang_switch_href": "index_ja.html",
        "hide_panel": "☰ Hide Panel",
        "show_panel": "☰ Show Panel",
        "basemap_section": "Base Map Layer",
        "bm_esri_topo": "Esri World Topo Map (Default)",
        "bm_google_road": "Google Maps (Roadmap)",
        "bm_google_hybrid": "Google Maps (Hybrid Satellite)",
        "bm_google_terrain": "Google Maps (Terrain)",
        "stats_section": "Filtered Māori Land Summary",
        "owners_unit": "Owners",
        "stat_visible_parcels": "Visible Parcels",
        "stat_total_area": "Total Area (ha)",
        "stat_std_blocks": "Standard Blocks",
        "stat_agg_blocks": "Aggregated Blocks",
        "owner_section": "Owner Numbers & Visibility Range",
        "status_hidden": "Hidden (Toggle ON to view)",
        "labels_in_view_suffix": "labels in view",
        "toggle_owner_title": "Show Owner Numbers on Map",
        "toggle_owner_sub": "Toggle on-map owner count labels (<code>is_rep = 1</code>)",
        "range_label": "Visible Owner Count Range (0 to All)",
        "all_label": "All",
        "min_label": "Min:",
        "max_label": "Max:",
        "reset_all": "Reset All",
        "min_slider_label": f"Min Owners Slider (0 → {MAX_OWNERS_IN_DATA:,})",
        "max_slider_label": f"Max Owners Slider (0 → {MAX_OWNERS_IN_DATA:,})",
        "preset_all": f"0 – All ({MAX_OWNERS_IN_DATA:,})",
        "chk_filter_parcels": "Filter map polygons by this owner range (as well as labels)",
        "chk_include_unrecorded": "Include parcels with unrecorded owner count when Min = 0",
        "bookmarks_section": "Quick Region Views",
        "bm_wide": "Wide Geothermal Zone",
        "bm_reporoa": "Reporoa Study Area",
        "bm_wairakei": "Wairakei / Tauhara / NAP",
        "bm_ohaaki": "Ohaaki / Ngatamariki",
        "bm_mokai": "Mokai",
        "search_section": "Search & Ownership Category",
        "search_label": "Search Title No, Block Description, or Plant",
        "search_placeholder": "e.g. 423025, Wharetoto, Tauhara, Ohaaki...",
        "no_matches": "No matching titles or plants found",
        "ownership_cat_label": "Ownership Category Filter",
        "opt_all_maori": "All Māori Land Titles (Standard + Aggregated)",
        "opt_std_only": "Māori Land — Standard Only (is_aggregated = 0)",
        "opt_agg_only": "Aggregated Ownership Only (is_aggregated = 1)",
        "maori_opacity_label": "Māori Land Fill Opacity",
        "primary_layers_section": "Primary Map Layers",
        "lyr_plants": "Geothermal Power Stations (6)",
        "lyr_plant_labels": "Geothermal Station Name Labels",
        "lyr_std_maori": "Māori Land (Standard)",
        "lyr_agg_maori": "Aggregated Ownership (Shared Owners)",
        "rohe_section": "Iwi Areas of Interest (Rohe)",
        "btn_all": "All",
        "btn_none": "None",
        "rohe_opacity_label": "Rohe Fill Opacity",
        "legend_title": "Map Legend",
        "leg_std_maori": "Māori land (standard)",
        "leg_agg_maori": "Aggregated ownership (owners shared across blocks)",
        "leg_tahu": "Ngāti Tahu / Ngāti Whaoa rohe",
        "leg_tuwharetoa": "Ngāti Tūwharetoa rohe",
        "leg_plant": "Geothermal power station",
        "legend_note": "Numbers indicate owners recorded by the Māori Land Court (as at 30 June 2025), shown once per ownership group (<code>is_rep = 1</code>). Matched to LINZ property titles. Ownership does not establish customary authority or representation.",
        "rohe_prefix": "Rohe",
        "area_label": "Area",
        "tpk_code_label": "TPK Code",
        "source_label": "Source",
        "popup_title_prefix": "LINZ Title",
        "badge_agg": "Aggregated Ownership (Shared Owners)",
        "badge_std": "Māori Land (Standard)",
        "not_recorded": "Not recorded",
        "imputed_tag": "(Imputed)",
        "mlc_owners_label": "No. of Owners (MLC)",
        "linz_owners_label": "LINZ Title Owners",
        "parcel_area_label": "Parcel Area",
        "title_type_status_label": "Title Type / Status",
        "land_district_label": "Land District",
        "issue_date_label": "Issue Date",
        "ownership_group_label": "Ownership Group",
        "rep_block_label": "Representative Block",
        "rep_yes": "Yes (is_rep = 1)",
        "rep_no": "No (is_rep = 0)",
        "estate_des_label": "Estate Description",
        "tt_title": "Title",
        "tt_mlc_owners": "MLC Owners",
        "tt_agg": "Aggregated ownership",
        "tt_std": "Māori land (standard)",
        "plant_popup_suffix": "Geothermal Plant",
        "plant_cap_label": "Installed Capacity",
        "plant_tech_label": "Technology Type",
        "plant_op_label": "Operator",
        "plant_nztm_label": "NZTM (EPSG:2193)",
        "plant_wgs_label": "WGS84 (Lat, Lon)",
        "search_plant_prefix": "Plant",
        "search_title_prefix": "Title",
        "search_owners_suffix": "owners",
    },
    "ja": {
        "html_lang": "ja",
        "page_title": "タウポ・レポロア地熱地帯＆マオリ土地 インタラクティブGISマップ",
        "header_title": "地熱発電所・マオリ土地・ロヘ GISマップ",
        "header_sub": "タウポ〜レポロア地熱地帯 • LINZ不動産タイトル + マオリ土地裁判所 (2025) & TPK Iwi ロヘ",
        "lang_switch_label": "🇬🇧 English Version",
        "lang_switch_href": "index.html",
        "hide_panel": "☰ パネルを隠す",
        "show_panel": "☰ パネルを表示",
        "basemap_section": "背景地図レイヤー (Base Map)",
        "bm_esri_topo": "Esri World Topo Map（標準地形図・デフォルト）",
        "bm_google_road": "Google マップ（標準道路地図）",
        "bm_google_hybrid": "Google マップ（衛星写真＋地名ハイブリッド）",
        "bm_google_terrain": "Google マップ（地形図 Terrain）",
        "stats_section": "表示中のマオリ土地 集計サマリー",
        "owners_unit": "人（所有者数）",
        "stat_visible_parcels": "表示区画数",
        "stat_total_area": "合計面積 (ha)",
        "stat_std_blocks": "通常所有区画",
        "stat_agg_blocks": "集約所有区画 (オレンジ)",
        "owner_section": "所有者数の表示切替・フィルタ範囲",
        "status_hidden": "非表示（スイッチONで表示）",
        "labels_in_view_suffix": "件のラベルを表示中",
        "toggle_owner_title": "地図上に所有者数ラベルを表示",
        "toggle_owner_sub": "所有者グループ代表区画（<code>is_rep = 1</code>）の所有者数を表示",
        "range_label": "表示する所有者数の範囲（0 〜 全件）",
        "all_label": "全件",
        "min_label": "最小:",
        "max_label": "最大:",
        "reset_all": "全件リセット",
        "min_slider_label": f"最小所有者数スライダー (0 → {MAX_OWNERS_IN_DATA:,})",
        "max_slider_label": f"最大所有者数スライダー (0 → {MAX_OWNERS_IN_DATA:,})",
        "preset_all": f"0 – 全件 ({MAX_OWNERS_IN_DATA:,})",
        "chk_filter_parcels": "ラベルだけでなく区画ポリゴン自体もこの所有者数範囲で絞り込む",
        "chk_include_unrecorded": "最小値=0 のとき所有者数未記録（空欄）の区画も含める",
        "bookmarks_section": "主要エリアへ移動（クイックビュー）",
        "bm_wide": "広域地熱地帯（全体）",
        "bm_reporoa": "レポロア調査対象地域",
        "bm_wairakei": "Wairakei / Tauhara / NAP",
        "bm_ohaaki": "Ohaaki / Ngatamariki",
        "bm_mokai": "Mokai（モカイ）",
        "search_section": "検索 ＆ 所有形態フィルタ",
        "search_label": "タイトル番号・区画名・地熱発電所を検索",
        "search_placeholder": "例: 423025, Wharetoto, Tauhara, Ohaaki...",
        "no_matches": "該当するタイトルまたは発電所が見つかりません",
        "ownership_cat_label": "所有形態による絞り込み",
        "opt_all_maori": "すべてのマオリ土地（通常所有 ＋ 集約所有）",
        "opt_std_only": "マオリ土地 — 通常所有のみ (is_aggregated = 0)",
        "opt_agg_only": "集約所有権のみ（複数区画で所有者を共有, is_aggregated = 1）",
        "maori_opacity_label": "マオリ土地ポリゴンの不透明度",
        "primary_layers_section": "主要マップレイヤー",
        "lyr_plants": "地熱発電所 (6箇所)",
        "lyr_plant_labels": "地熱発電所の名称ラベル",
        "lyr_std_maori": "マオリ土地（通常所有）",
        "lyr_agg_maori": "集約所有権（複数区画で所有者リストを共有）",
        "rohe_section": "イウィ関心領域（Rohe / ロヘ）",
        "btn_all": "すべて表示",
        "btn_none": "すべて非表示",
        "rohe_opacity_label": "ロヘ領域の不透明度",
        "legend_title": "凡例 (Legend)",
        "leg_std_maori": "マオリ土地（通常所有）",
        "leg_agg_maori": "集約所有権（複数区画で所有者リストを共有）",
        "leg_tahu": "Ngāti Tahu / Ngāti Whaoa ロヘ",
        "leg_tuwharetoa": "Ngāti Tūwharetoa ロヘ",
        "leg_plant": "地熱発電所",
        "legend_note": "数字はマオリ土地裁判所（2025年6月30日時点）に記録された所有者数を示す。オレンジの区画は所有権が集約されており、複数区画で1つの所有者リストを共有しているため、数字は所有者グループごとに1つ表示している（<code>is_rep = 1</code>）。LINZの区画・タイトル境界と照合。所有は慣習的権限や代表性を立証するものではない。",
        "rohe_prefix": "ロヘ (Rohe)",
        "area_label": "面積",
        "tpk_code_label": "TPKコード",
        "source_label": "出典",
        "popup_title_prefix": "LINZ タイトル番号",
        "badge_agg": "集約所有権（複数区画で所有者共有）",
        "badge_std": "マオリ土地（通常所有）",
        "not_recorded": "未記録",
        "imputed_tag": "（補完値）",
        "mlc_owners_label": "所有者数（マオリ土地裁判所）",
        "linz_owners_label": "LINZ 登録所有者数",
        "parcel_area_label": "区画面積",
        "title_type_status_label": "種別 / ステータス",
        "land_district_label": "土地登記管轄区",
        "issue_date_label": "発行日",
        "ownership_group_label": "所有者グループID",
        "rep_block_label": "グループ代表区画",
        "rep_yes": "はい (is_rep = 1)",
        "rep_no": "いいえ (is_rep = 0)",
        "estate_des_label": "法定不動産記述 (Estate Description)",
        "tt_title": "タイトル",
        "tt_mlc_owners": "MLC所有者数",
        "tt_agg": "集約所有権（複数区画共有）",
        "tt_std": "マオリ土地（通常所有）",
        "plant_popup_suffix": "地熱発電所",
        "plant_cap_label": "設備容量 (出力)",
        "plant_tech_label": "発電方式",
        "plant_op_label": "運営事業者",
        "plant_nztm_label": "NZTM座標 (EPSG:2193)",
        "plant_wgs_label": "緯度・経度 (WGS84)",
        "search_plant_prefix": "発電所",
        "search_title_prefix": "タイトル",
        "search_owners_suffix": "人",
    },
}

def build_html_app(lang_code: str) -> str:
    t = UI_TEXT[lang_code]
    active_en = "active" if lang_code == "en" else ""
    active_ja = "active" if lang_code == "ja" else ""
    return f"""<!DOCTYPE html>
<html lang="{t['html_lang']}">
<head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>{t['page_title']}</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
    <style>
        :root {{
            --bg-panel: #ffffff;
            --bg-subtle: #f8fafc;
            --border-color: #e2e8f0;
            --text-main: #0f172a;
            --text-muted: #475569;
            --accent: #0284c7;
            --std-maori: {STYLE_STANDARD_MAORI["fillColor"]};
            --agg-maori: {STYLE_AGGREGATED_MAORI["fillColor"]};
            --plant-red: {PLANT_MARKER_COLOR};
        }}
        * {{ box-sizing: border-box; }}
        html, body {{
            margin: 0;
            padding: 0;
            width: 100%;
            height: 100%;
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "Hiragino Sans", "Yu Gothic UI", "Meiryo", Roboto, sans-serif;
            color: var(--text-main);
            overflow: hidden;
        }}
        #app-container {{
            display: flex;
            width: 100%;
            height: 100%;
            position: relative;
        }}
        #sidebar {{
            width: 395px;
            min-width: 330px;
            max-width: 92vw;
            height: 100%;
            background: var(--bg-panel);
            border-right: 1px solid var(--border-color);
            display: flex;
            flex-direction: column;
            z-index: 1000;
            box-shadow: 2px 0 12px rgba(15, 23, 42, 0.08);
            transition: margin-left 0.25s ease;
        }}
        #sidebar.collapsed {{
            margin-left: -395px;
        }}
        #sidebar-toggle {{
            position: absolute;
            top: 12px;
            left: 407px;
            z-index: 1001;
            background: #ffffff;
            border: 1px solid var(--border-color);
            border-radius: 6px;
            padding: 7px 11px;
            font-size: 13px;
            font-weight: 600;
            color: var(--text-main);
            cursor: pointer;
            box-shadow: 0 2px 6px rgba(0,0,0,0.12);
            transition: left 0.25s ease;
        }}
        #sidebar.collapsed + #sidebar-toggle {{
            left: 12px;
        }}
        .sidebar-header {{
            padding: 13px 16px;
            background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%);
            color: #ffffff;
        }}
        .header-top-row {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            gap: 8px;
        }}
        .sidebar-header h1 {{
            margin: 0;
            font-size: 14.5px;
            font-weight: 700;
            letter-spacing: 0.2px;
        }}
        .lang-toggle {{
            display: inline-flex;
            align-items: center;
            background: rgba(15, 23, 42, 0.45);
            border: 1px solid rgba(255, 255, 255, 0.28);
            border-radius: 20px;
            padding: 2px;
            gap: 2px;
            box-shadow: inset 0 1px 3px rgba(0, 0, 0, 0.25);
            flex-shrink: 0;
        }}
        .lang-toggle-opt {{
            padding: 3px 10px;
            font-size: 11px;
            font-weight: 700;
            letter-spacing: 0.5px;
            text-decoration: none;
            border-radius: 16px;
            color: rgba(255, 255, 255, 0.72);
            line-height: 1.25;
            transition: all 0.18s ease;
            user-select: none;
        }}
        .lang-toggle-opt:hover {{
            color: #ffffff;
            background: rgba(255, 255, 255, 0.18);
        }}
        .lang-toggle-opt.active {{
            background: #ffffff;
            color: #0f172a;
            box-shadow: 0 1px 3px rgba(0, 0, 0, 0.25);
            cursor: default;
            pointer-events: none;
        }}
        .sidebar-header p {{
            margin: 5px 0 0 0;
            font-size: 11px;
            color: #cbd5e1;
            line-height: 1.35;
        }}
        .sidebar-content {{
            flex: 1;
            overflow-y: auto;
            padding: 12px 14px 24px 14px;
        }}
        .section-card {{
            background: var(--bg-subtle);
            border: 1px solid var(--border-color);
            border-radius: 8px;
            padding: 11px 12px;
            margin-bottom: 11px;
        }}
        .section-title {{
            font-size: 11.5px;
            font-weight: 700;
            text-transform: uppercase;
            letter-spacing: 0.4px;
            color: #334155;
            margin: 0 0 8px 0;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}
        /* Stats Grid */
        .stats-grid {{
            display: grid;
            grid-template-columns: repeat(2, 1fr);
            gap: 7px;
        }}
        .stat-box {{
            background: #ffffff;
            border: 1px solid var(--border-color);
            border-radius: 6px;
            padding: 7px 9px;
        }}
        .stat-label {{
            font-size: 10.5px;
            color: var(--text-muted);
            margin-bottom: 2px;
        }}
        .stat-value {{
            font-size: 14.5px;
            font-weight: 700;
            color: var(--text-main);
        }}
        .stat-value.orange {{ color: #d97706; }}
        .stat-value.slate {{ color: #475569; }}
        /* Bookmarks & Presets */
        .bookmark-grid {{
            display: flex;
            flex-wrap: wrap;
            gap: 5px;
        }}
        .btn-chip {{
            background: #ffffff;
            border: 1px solid #cbd5e1;
            border-radius: 5px;
            padding: 4px 8px;
            font-size: 11px;
            font-weight: 500;
            color: #1e293b;
            cursor: pointer;
            transition: all 0.15s;
        }}
        .btn-chip:hover {{
            background: #f1f5f9;
            border-color: #94a3b8;
        }}
        .btn-chip.active {{
            background: #0284c7;
            color: #ffffff;
            border-color: #0284c7;
        }}
        /* Prominent Switch Box for Owner Numbers */
        .toggle-banner {{
            display: flex;
            align-items: center;
            justify-content: space-between;
            background: #ffffff;
            border: 1.5px solid #cbd5e1;
            border-radius: 7px;
            padding: 8px 10px;
            margin-bottom: 10px;
            cursor: pointer;
            transition: all 0.15s;
        }}
        .toggle-banner.active {{
            border-color: #0284c7;
            background: #f0f9ff;
        }}
        .toggle-banner-title {{
            font-size: 12.5px;
            font-weight: 700;
            color: #0f172a;
        }}
        .toggle-banner-sub {{
            font-size: 10.5px;
            color: #64748b;
            margin-top: 1px;
        }}
        /* Controls & Inputs */
        .control-row {{
            margin-bottom: 10px;
        }}
        .control-row:last-child {{
            margin-bottom: 0;
        }}
        .control-label {{
            display: flex;
            justify-content: space-between;
            align-items: center;
            font-size: 12px;
            font-weight: 500;
            margin-bottom: 4px;
            color: #1e293b;
        }}
        .search-input, .select-input {{
            width: 100%;
            padding: 6px 9px;
            font-size: 12.5px;
            border: 1px solid #cbd5e1;
            border-radius: 5px;
            background: #ffffff;
        }}
        .num-range-row {{
            display: flex;
            align-items: center;
            gap: 6px;
            margin-top: 4px;
        }}
        .num-input {{
            width: 82px;
            padding: 4px 6px;
            font-size: 12px;
            border: 1px solid #cbd5e1;
            border-radius: 4px;
            text-align: right;
            font-weight: 600;
        }}
        .search-results {{
            max-height: 150px;
            overflow-y: auto;
            margin-top: 5px;
            border: 1px solid var(--border-color);
            border-radius: 5px;
            background: #ffffff;
            display: none;
        }}
        .search-item {{
            padding: 6px 8px;
            font-size: 11.5px;
            border-bottom: 1px solid #f1f5f9;
            cursor: pointer;
        }}
        .search-item:hover {{
            background: #e0f2fe;
        }}
        input[type="range"] {{
            width: 100%;
            accent-color: var(--accent);
            cursor: pointer;
        }}
        /* Layer Checkboxes */
        .layer-item {{
            display: flex;
            align-items: center;
            gap: 8px;
            padding: 4px 0;
            font-size: 12.5px;
            cursor: pointer;
        }}
        .layer-item input {{
            cursor: pointer;
            width: 14px;
            height: 14px;
            accent-color: var(--accent);
        }}
        .swatch {{
            width: 14px;
            height: 14px;
            border-radius: 3px;
            display: inline-block;
            flex-shrink: 0;
        }}
        .swatch-triangle {{
            width: 0;
            height: 0;
            border-left: 7px solid transparent;
            border-right: 7px solid transparent;
            border-bottom: 13px solid var(--plant-red);
            display: inline-block;
            flex-shrink: 0;
        }}
        .rohe-scroll {{
            max-height: 165px;
            overflow-y: auto;
            padding-right: 4px;
            margin-top: 4px;
            border-top: 1px dashed #cbd5e1;
            padding-top: 6px;
        }}
        /* Map Area */
        #map {{
            flex: 1;
            height: 100%;
            background: #e2e8f0;
        }}
        /* Floating Map Legend & Readout */
        .map-legend {{
            background: rgba(255, 255, 255, 0.95);
            padding: 10px 13px;
            border-radius: 7px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.15);
            border: 1px solid #cbd5e1;
            font-size: 12px;
            line-height: 1.45;
            max-width: 295px;
        }}
        .map-legend h4 {{
            margin: 0 0 6px 0;
            font-size: 12.5px;
            font-weight: 700;
            color: #0f172a;
        }}
        .legend-row {{
            display: flex;
            align-items: center;
            gap: 8px;
            margin-bottom: 5px;
        }}
        .legend-note {{
            margin-top: 7px;
            padding-top: 6px;
            border-top: 1px solid #e2e8f0;
            font-size: 10.5px;
            color: #475569;
            line-height: 1.35;
        }}
        .coord-box {{
            background: rgba(15, 23, 42, 0.85);
            color: #f8fafc;
            padding: 5px 10px;
            border-radius: 5px;
            font-size: 11.5px;
            font-family: monospace;
            box-shadow: 0 1px 5px rgba(0,0,0,0.2);
        }}
        /* Owner Count Map Labels */
        .owner-label-icon {{
            background: transparent;
            border: none;
        }}
        .owner-label-text {{
            font-size: 11px;
            font-weight: 700;
            color: #0f172a;
            text-shadow: -1.5px -1.5px 0 #fff, 1.5px -1.5px 0 #fff, -1.5px 1.5px 0 #fff, 1.5px 1.5px 0 #fff, 0px 0px 4px #fff;
            white-space: nowrap;
            pointer-events: none;
            transform: translate(-50%, -50%);
            display: inline-block;
        }}
        .owner-label-text.agg {{
            color: #78350f;
            text-shadow: -1.5px -1.5px 0 #fef3c7, 1.5px -1.5px 0 #fef3c7, -1.5px 1.5px 0 #fef3c7, 1.5px 1.5px 0 #fef3c7;
        }}
        /* Geothermal Plant Labels */
        .plant-label-tooltip {{
            background: rgba(255, 255, 255, 0.92) !important;
            border: 1.5px solid #dc2626 !important;
            border-radius: 4px !important;
            color: #991b1b !important;
            font-weight: 700 !important;
            font-size: 11.5px !important;
            padding: 2px 6px !important;
            box-shadow: 0 1px 4px rgba(0,0,0,0.2) !important;
        }}
        /* Popup Table Styling */
        .popup-card {{
            min-width: 255px;
            font-size: 12px;
        }}
        .popup-card h3 {{
            margin: 0 0 6px 0;
            font-size: 13.5px;
            padding-bottom: 5px;
            border-bottom: 2px solid #0284c7;
            color: #0f172a;
        }}
        .popup-badge {{
            display: inline-block;
            padding: 2px 6px;
            border-radius: 4px;
            font-size: 10.5px;
            font-weight: 700;
            margin-bottom: 6px;
        }}
        .badge-std {{ background: #e2e8f0; color: #1e293b; }}
        .badge-agg {{ background: #fef3c7; color: #92400e; border: 1px solid #f59e0b; }}
        .popup-table {{
            width: 100%;
            border-collapse: collapse;
        }}
        .popup-table td {{
            padding: 3px 4px;
            border-bottom: 1px solid #f1f5f9;
            vertical-align: top;
        }}
        .popup-table td.k {{
            color: #64748b;
            font-weight: 600;
            width: 44%;
        }}
        .popup-table td.v {{
            color: #0f172a;
            word-break: break-word;
        }}
    </style>
</head>
<body>
<div id="app-container">
    <aside id="sidebar">
        <div class="sidebar-header">
            <div class="header-top-row">
                <h1>{t['header_title']}</h1>
                <div class="lang-toggle" role="group" aria-label="Language selector">
                    <a href="index.html" class="lang-toggle-opt {active_en}" title="English">EN</a>
                    <a href="index_ja.html" class="lang-toggle-opt {active_ja}" title="日本語">JA</a>
                </div>
            </div>
            <p>{t['header_sub']}</p>
        </div>
        <div class="sidebar-content">
            <!-- Base Map Selector (Esri World Topo Default + 3 Google Maps Layers Only) -->
            <div class="section-card">
                <div class="section-title">{t['basemap_section']}</div>
                <select id="basemap-select" class="select-input" onchange="switchBaseMap(this.value)">
                    <option value="{t['bm_esri_topo']}" selected>{t['bm_esri_topo']}</option>
                    <option value="{t['bm_google_road']}">{t['bm_google_road']}</option>
                    <option value="{t['bm_google_hybrid']}">{t['bm_google_hybrid']}</option>
                    <option value="{t['bm_google_terrain']}">{t['bm_google_terrain']}</option>
                </select>
            </div>

            <!-- Live Statistics -->
            <div class="section-card">
                <div class="section-title">
                    <span>{t['stats_section']}</span>
                    <span id="filter-status-badge" style="font-size:10px; color:#0284c7;">0 – {MAX_OWNERS_IN_DATA:,} {t['owners_unit']}</span>
                </div>
                <div class="stats-grid">
                    <div class="stat-box">
                        <div class="stat-label">{t['stat_visible_parcels']}</div>
                        <div class="stat-value" id="stat-total-parcels">-</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-label">{t['stat_total_area']}</div>
                        <div class="stat-value" id="stat-total-area">-</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-label">{t['stat_std_blocks']}</div>
                        <div class="stat-value slate" id="stat-std-parcels">-</div>
                    </div>
                    <div class="stat-box">
                        <div class="stat-label">{t['stat_agg_blocks']}</div>
                        <div class="stat-value orange" id="stat-agg-parcels">-</div>
                    </div>
                </div>
            </div>

            <!-- Owner Numbers Visibility & Range Control (0 to All / 17,737) -->
            <div class="section-card">
                <div class="section-title">
                    <span>{t['owner_section']}</span>
                    <span id="visible-labels-count" style="font-size:10.5px; color:#475569;">{t['status_hidden']}</span>
                </div>

                <!-- Prominent Toggle Switch for Owner Numbers on Map -->
                <label class="toggle-banner" id="owner-label-banner">
                    <div>
                        <div class="toggle-banner-title">{t['toggle_owner_title']}</div>
                        <div class="toggle-banner-sub">{t['toggle_owner_sub']}</div>
                    </div>
                    <input type="checkbox" id="toggle-owner-labels" onchange="onToggleOwnerLabels(this.checked)" style="width:18px; height:18px; accent-color:#0284c7; cursor:pointer;" />
                </label>

                <!-- Range of Owners Visible (0 to 17,737 / All) -->
                <div class="control-row">
                    <div class="control-label">
                        <span>{t['range_label']}</span>
                        <strong id="owner-range-readout">0 – {MAX_OWNERS_IN_DATA:,} ({t['all_label']})</strong>
                    </div>
                    <div class="num-range-row">
                        <span style="font-size:11px; color:#475569;">{t['min_label']}</span>
                        <input type="number" id="min-owners-input" class="num-input" min="0" max="{MAX_OWNERS_IN_DATA}" value="0" onchange="syncOwnerRangeFromInputs()" />
                        <span style="font-size:11px; color:#475569;">{t['max_label']}</span>
                        <input type="number" id="max-owners-input" class="num-input" min="0" max="{MAX_OWNERS_IN_DATA}" value="{MAX_OWNERS_IN_DATA}" onchange="syncOwnerRangeFromInputs()" />
                        <button class="btn-chip" onclick="setOwnerPreset(0, {MAX_OWNERS_IN_DATA})">{t['reset_all']}</button>
                    </div>
                    <div style="margin-top:7px;">
                        <div style="display:flex; justify-content:space-between; font-size:11px; color:#475569;">
                            <span>{t['min_slider_label']}</span>
                        </div>
                        <input type="range" id="min-owners-slider" min="0" max="{MAX_OWNERS_IN_DATA}" step="1" value="0" oninput="syncOwnerRangeFromSliders()" />
                        <div style="display:flex; justify-content:space-between; font-size:11px; color:#475569; margin-top:2px;">
                            <span>{t['max_slider_label']}</span>
                        </div>
                        <input type="range" id="max-owners-slider" min="0" max="{MAX_OWNERS_IN_DATA}" step="1" value="{MAX_OWNERS_IN_DATA}" oninput="syncOwnerRangeFromSliders()" />
                    </div>
                    <div class="bookmark-grid" style="margin-top:6px;">
                        <button class="btn-chip" onclick="setOwnerPreset(0, {MAX_OWNERS_IN_DATA})">{t['preset_all']}</button>
                        <button class="btn-chip" onclick="setOwnerPreset(1, {MAX_OWNERS_IN_DATA})">≥ 1</button>
                        <button class="btn-chip" onclick="setOwnerPreset(10, {MAX_OWNERS_IN_DATA})">≥ 10</button>
                        <button class="btn-chip" onclick="setOwnerPreset(100, {MAX_OWNERS_IN_DATA})">≥ 100</button>
                        <button class="btn-chip" onclick="setOwnerPreset(500, {MAX_OWNERS_IN_DATA})">≥ 500</button>
                        <button class="btn-chip" onclick="setOwnerPreset(1000, {MAX_OWNERS_IN_DATA})">≥ 1,000</button>
                        <button class="btn-chip" onclick="setOwnerPreset(5000, {MAX_OWNERS_IN_DATA})">≥ 5,000</button>
                    </div>
                </div>

                <div class="control-row" style="margin-top:8px; padding-top:6px; border-top:1px dashed #cbd5e1;">
                    <label class="layer-item" style="font-size:11.5px;">
                        <input type="checkbox" id="apply-range-to-parcels" checked onchange="applyFilters()" />
                        <span>{t['chk_filter_parcels']}</span>
                    </label>
                    <label class="layer-item" style="font-size:11.5px;">
                        <input type="checkbox" id="include-unrecorded-owners" checked onchange="applyFilters()" />
                        <span>{t['chk_include_unrecorded']}</span>
                    </label>
                </div>
            </div>

            <!-- Quick Region Bookmarks -->
            <div class="section-card">
                <div class="section-title">{t['bookmarks_section']}</div>
                <div class="bookmark-grid">
                    <button class="btn-chip active" onclick="flyToBookmark(-38.55, 176.18, 10, this)">{t['bm_wide']}</button>
                    <button class="btn-chip" onclick="flyToBookmark(-38.435, 176.37, 12, this)">{t['bm_reporoa']}</button>
                    <button class="btn-chip" onclick="flyToBookmark(-38.63, 176.14, 12, this)">{t['bm_wairakei']}</button>
                    <button class="btn-chip" onclick="flyToBookmark(-38.535, 176.33, 12, this)">{t['bm_ohaaki']}</button>
                    <button class="btn-chip" onclick="flyToBookmark(-38.535, 175.92, 12, this)">{t['bm_mokai']}</button>
                </div>
            </div>

            <!-- Search & Parcel Filters -->
            <div class="section-card">
                <div class="section-title">{t['search_section']}</div>
                <div class="control-row">
                    <div class="control-label"><span>{t['search_label']}</span></div>
                    <input type="text" id="search-box" class="search-input" placeholder="{t['search_placeholder']}" oninput="handleSearch(this.value)" />
                    <div id="search-results" class="search-results"></div>
                </div>
                <div class="control-row">
                    <div class="control-label"><span>{t['ownership_cat_label']}</span></div>
                    <select id="ownership-filter" class="select-input" onchange="applyFilters()">
                        <option value="all">{t['opt_all_maori']}</option>
                        <option value="standard">{t['opt_std_only']}</option>
                        <option value="aggregated">{t['opt_agg_only']}</option>
                    </select>
                </div>
                <div class="control-row">
                    <div class="control-label">
                        <span>{t['maori_opacity_label']}</span>
                        <strong id="maori-opacity-val">60%</strong>
                    </div>
                    <input type="range" id="maori-opacity" min="10" max="95" step="5" value="60" oninput="updateMaoriOpacity(this.value)" />
                </div>
            </div>

            <!-- Map Layers Control -->
            <div class="section-card">
                <div class="section-title">{t['primary_layers_section']}</div>
                <label class="layer-item">
                    <input type="checkbox" id="toggle-plants" checked onchange="toggleLayerGroup('plants', this.checked)" />
                    <span class="swatch-triangle"></span>
                    <span>{t['lyr_plants']}</span>
                </label>
                <label class="layer-item">
                    <input type="checkbox" id="toggle-plant-labels" checked onchange="togglePlantLabels(this.checked)" />
                    <span class="swatch" style="background:#fee2e2; border:1.5px solid #dc2626;"></span>
                    <span>{t['lyr_plant_labels']}</span>
                </label>
                <label class="layer-item">
                    <input type="checkbox" id="toggle-std-maori" checked onchange="applyFilters()" />
                    <span class="swatch" style="background:{STYLE_STANDARD_MAORI["fillColor"]}; border:1.5px solid {STYLE_STANDARD_MAORI["color"]};"></span>
                    <span>{t['lyr_std_maori']}</span>
                </label>
                <label class="layer-item">
                    <input type="checkbox" id="toggle-agg-maori" checked onchange="applyFilters()" />
                    <span class="swatch" style="background:{STYLE_AGGREGATED_MAORI["fillColor"]}; border:1.5px solid {STYLE_AGGREGATED_MAORI["color"]};"></span>
                    <span>{t['lyr_agg_maori']}</span>
                </label>
            </div>

            <!-- Iwi Rohe Layers -->
            <div class="section-card">
                <div class="section-title">
                    <span>{t['rohe_section']}</span>
                    <div>
                        <button class="btn-chip" style="padding:2px 6px; font-size:10px;" onclick="setAllRohe(true)">{t['btn_all']}</button>
                        <button class="btn-chip" style="padding:2px 6px; font-size:10px;" onclick="setAllRohe(false)">{t['btn_none']}</button>
                    </div>
                </div>
                <div id="primary-rohe-list"></div>
                <div class="rohe-scroll" id="secondary-rohe-list"></div>
                <div class="control-row" style="margin-top:8px;">
                    <div class="control-label">
                        <span>{t['rohe_opacity_label']}</span>
                        <strong id="rohe-opacity-val">15%</strong>
                    </div>
                    <input type="range" id="rohe-opacity" min="0" max="60" step="5" value="15" oninput="updateRoheOpacity(this.value)" />
                </div>
            </div>
        </div>
    </aside>
    <button id="sidebar-toggle" onclick="toggleSidebar()">{t['hide_panel']}</button>
    <main id="map"></main>
</div>

<script>
// ============================================================================
// EMBEDDED PROCESSED DATASETS (Generated from Jupyter Notebook)
// ============================================================================
const LANG = {json.dumps(lang_code)};
const I18N = {json.dumps(t, ensure_ascii=False)};
const APP_CONFIG = {json.dumps(app_config)};
const PLANTS_DATA = {json.dumps(plants_list, ensure_ascii=False)};
const ROHE_LAYERS_DATA = {json.dumps(rohe_layers_data, ensure_ascii=False)};
const MAORI_GEOJSON = {json.dumps(maori_geojson_obj, ensure_ascii=False)};

// ============================================================================
// INITIALIZE LEAFLET MAP & BASE LAYERS (Esri World Topo Default + 3 Google Maps)
// ============================================================================
const map = L.map('map', {{
    center: APP_CONFIG.center,
    zoom: APP_CONFIG.zoom,
    preferCanvas: true,
    zoomControl: false
}});

L.control.zoom({{ position: 'topright' }}).addTo(map);
L.control.scale({{ position: 'bottomright', metric: true, imperial: false }}).addTo(map);

const baseMaps = {{
    [I18N.bm_esri_topo]: L.tileLayer(APP_CONFIG.tileUrls.esriTopo, {{
        maxZoom: 19,
        attribution: 'Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ, TomTom, Intermap, iPC, USGS, FAO, NPS, NRCAN, GeoBase, Kadaster NL, Ordnance Survey, Esri Japan, METI, Esri China (Hong Kong), and the GIS User Community'
    }}),
    [I18N.bm_google_road]: L.tileLayer(APP_CONFIG.tileUrls.googleRoadmap, {{
        maxZoom: 20,
        attribution: '&copy; Google Maps'
    }}),
    [I18N.bm_google_hybrid]: L.tileLayer(APP_CONFIG.tileUrls.googleHybrid, {{
        maxZoom: 20,
        attribution: '&copy; Google Maps'
    }}),
    [I18N.bm_google_terrain]: L.tileLayer(APP_CONFIG.tileUrls.googleTerrain, {{
        maxZoom: 20,
        attribution: '&copy; Google Maps'
    }})
}};

// Set Esri World Topo Map as the default base layer
let activeBaseLayer = baseMaps[I18N.bm_esri_topo];
activeBaseLayer.addTo(map);
L.control.layers(baseMaps, null, {{ position: 'topright', collapsed: true }}).addTo(map);

function switchBaseMap(name) {{
    if (baseMaps[name]) {{
        if (activeBaseLayer) map.removeLayer(activeBaseLayer);
        activeBaseLayer = baseMaps[name];
        activeBaseLayer.addTo(map);
    }}
}}

map.on('baselayerchange', (e) => {{
    activeBaseLayer = e.layer;
    const sel = document.getElementById('basemap-select');
    if (sel) sel.value = e.name;
}});

// Custom panes for clean z-index ordering: Rohe (bottom) -> Māori Land (middle) -> Labels & Plants (top)
map.createPane('rohePane');
map.getPane('rohePane').style.zIndex = 350;
map.createPane('maoriPane');
map.getPane('maoriPane').style.zIndex = 420;
map.createPane('plantPane');
map.getPane('plantPane').style.zIndex = 650;

const sharedCanvas = L.canvas({{ padding: 0.5, pane: 'maoriPane' }});
const roheCanvas = L.canvas({{ padding: 0.5, pane: 'rohePane' }});

// ============================================================================
// 1. IWI ROHE LAYERS
// ============================================================================
const roheLayerObjects = {{}};
const primaryRoheContainer = document.getElementById('primary-rohe-list');
const secondaryRoheContainer = document.getElementById('secondary-rohe-list');

ROHE_LAYERS_DATA.forEach((item) => {{
    const displayName = (LANG === 'ja' && item.name_ja) ? item.name_ja : item.name;
    const layer = L.geoJSON(item.geometry, {{
        pane: 'rohePane',
        renderer: roheCanvas,
        style: () => ({{
            color: item.style.color,
            fillColor: item.style.fillColor,
            fillOpacity: item.style.fillOpacity,
            weight: item.style.weight,
            dashArray: item.style.dashArray
        }}),
        onEachFeature: (feat, lyr) => {{
            lyr.bindTooltip(`<strong>${{I18N.rohe_prefix}}: ${{displayName}}</strong><br/>${{I18N.area_label}}: ${{item.area_km2.toLocaleString()}} km²`, {{ sticky: true }});
            lyr.bindPopup(`
                <div class="popup-card">
                    <h3>${{displayName}}</h3>
                    <table class="popup-table">
                        <tr><td class="k">${{I18N.tpk_code_label}}</td><td class="v">${{item.tpk_code}}</td></tr>
                        <tr><td class="k">${{I18N.area_label}}</td><td class="v">${{item.area_km2.toLocaleString()}} km²</td></tr>
                        <tr><td class="k">${{I18N.source_label}}</td><td class="v">${{item.source}}</td></tr>
                    </table>
                </div>
            `);
        }}
    }});

    roheLayerObjects[item.name] = {{ layer, config: item }};
    if (item.default_on) {{
        layer.addTo(map);
    }}

    const labelEl = document.createElement('label');
    labelEl.className = 'layer-item';
    labelEl.innerHTML = `
        <input type="checkbox" class="rohe-checkbox" data-name="${{item.name}}" ${{item.default_on ? 'checked' : ''}} />
        <span class="swatch" style="background:${{item.style.fillColor}}44; border:2px dashed ${{item.style.color}};"></span>
        <span style="flex:1;">${{displayName}}</span>
    `;
    labelEl.querySelector('input').addEventListener('change', (e) => {{
        if (e.target.checked) {{
            layer.addTo(map);
        }} else {{
            map.removeLayer(layer);
        }}
    }});

    if (item.category === 'primary' || item.category === 'overlap') {{
        primaryRoheContainer.appendChild(labelEl);
    }} else {{
        secondaryRoheContainer.appendChild(labelEl);
    }}
}});

function setAllRohe(enable) {{
    document.querySelectorAll('.rohe-checkbox').forEach(cb => {{
        cb.checked = enable;
        const name = cb.getAttribute('data-name');
        const obj = roheLayerObjects[name];
        if (obj) {{
            if (enable) obj.layer.addTo(map);
            else map.removeLayer(obj.layer);
        }}
    }});
}}

function updateRoheOpacity(val) {{
    document.getElementById('rohe-opacity-val').textContent = val + '%';
    const fillOp = parseFloat(val) / 100.0;
    Object.values(roheLayerObjects).forEach(obj => {{
        obj.layer.setStyle({{ fillOpacity: fillOp }});
    }});
}}

// ============================================================================
// 2. MĀORI LAND PROPERTY TITLES & OWNER RANGE VISIBILITY (0 TO ALL / 17,737)
// ============================================================================
let currentMaoriOpacity = 0.60;
let maoriGeoJsonLayer = null;
const ownerLabelsLayerGroup = L.layerGroup().addTo(map);
const titleLookup = {{}};

const ownerLabelToggleEl = document.getElementById('toggle-owner-labels');
ownerLabelToggleEl.checked = Boolean(APP_CONFIG.showOwnerLabelsByDefault);
document.getElementById('owner-label-banner').classList.toggle('active', ownerLabelToggleEl.checked);

function onToggleOwnerLabels(checked) {{
    document.getElementById('owner-label-banner').classList.toggle('active', checked);
    refreshOwnerLabels();
}}

function getOwnerRangeValues() {{
    let minVal = parseInt(document.getElementById('min-owners-input').value, 10);
    let maxVal = parseInt(document.getElementById('max-owners-input').value, 10);
    if (isNaN(minVal) || minVal < 0) minVal = 0;
    if (isNaN(maxVal) || maxVal > APP_CONFIG.maxOwners) maxVal = APP_CONFIG.maxOwners;
    if (minVal > maxVal) {{
        const tmp = minVal;
        minVal = maxVal;
        maxVal = tmp;
    }}
    return {{ minVal, maxVal }};
}}

function updateOwnerRangeReadout(minVal, maxVal) {{
    const isAll = (minVal === 0 && maxVal >= APP_CONFIG.maxOwners);
    const text = isAll
        ? `0 – ${{APP_CONFIG.maxOwners.toLocaleString()}} (${{I18N.all_label}})`
        : `${{minVal.toLocaleString()}} – ${{maxVal.toLocaleString()}}`;
    document.getElementById('owner-range-readout').textContent = text;
    document.getElementById('filter-status-badge').textContent = text + ' ' + I18N.owners_unit;
}}

function syncOwnerRangeFromSliders() {{
    let minVal = parseInt(document.getElementById('min-owners-slider').value, 10) || 0;
    let maxVal = parseInt(document.getElementById('max-owners-slider').value, 10) || 0;
    if (minVal > maxVal) {{
        maxVal = minVal;
        document.getElementById('max-owners-slider').value = maxVal;
    }}
    document.getElementById('min-owners-input').value = minVal;
    document.getElementById('max-owners-input').value = maxVal;
    updateOwnerRangeReadout(minVal, maxVal);
    applyFilters();
}}

function syncOwnerRangeFromInputs() {{
    const {{ minVal, maxVal }} = getOwnerRangeValues();
    document.getElementById('min-owners-input').value = minVal;
    document.getElementById('max-owners-input').value = maxVal;
    document.getElementById('min-owners-slider').value = minVal;
    document.getElementById('max-owners-slider').value = maxVal;
    updateOwnerRangeReadout(minVal, maxVal);
    applyFilters();
}}

function setOwnerPreset(minVal, maxVal) {{
    document.getElementById('min-owners-input').value = minVal;
    document.getElementById('max-owners-input').value = maxVal;
    document.getElementById('min-owners-slider').value = minVal;
    document.getElementById('max-owners-slider').value = maxVal;
    updateOwnerRangeReadout(minVal, maxVal);
    applyFilters();
}}

function getMaoriParcelStyle(feature) {{
    const isAgg = feature.properties.is_aggregated === 1;
    if (isAgg) {{
        return {{
            color: APP_CONFIG.styleAggregatedMaori.color,
            fillColor: APP_CONFIG.styleAggregatedMaori.fillColor,
            fillOpacity: Math.min(0.95, currentMaoriOpacity + 0.12),
            weight: 1.4
        }};
    }}
    return {{
        color: APP_CONFIG.styleStandardMaori.color,
        fillColor: APP_CONFIG.styleStandardMaori.fillColor,
        fillOpacity: currentMaoriOpacity,
        weight: 1.0
    }};
}}

function buildMaoriPopupHtml(p) {{
    const isAgg = p.is_aggregated === 1;
    const badgeHtml = isAgg
        ? `<span class="popup-badge badge-agg">${{I18N.badge_agg}}</span>`
        : `<span class="popup-badge badge-std">${{I18N.badge_std}}</span>`;
    const mlcOwnersStr = p.mlc_owners !== null ? p.mlc_owners.toLocaleString() : I18N.not_recorded;
    const linzOwnersStr = p.number_own !== null ? p.number_own.toLocaleString() : '-';

    return `
        <div class="popup-card">
            <h3>${{I18N.popup_title_prefix}}: ${{p.title_no}}</h3>
            ${{badgeHtml}}
            <table class="popup-table">
                <tr><td class="k">${{I18N.mlc_owners_label}}</td><td class="v"><strong>${{mlcOwnersStr}}</strong>${{p.is_imputed ? ' <em>' + I18N.imputed_tag + '</em>' : ''}}</td></tr>
                <tr><td class="k">${{I18N.linz_owners_label}}</td><td class="v">${{linzOwnersStr}}</td></tr>
                <tr><td class="k">${{I18N.parcel_area_label}}</td><td class="v">${{p.area_ha.toLocaleString()}} ha</td></tr>
                <tr><td class="k">${{I18N.title_type_status_label}}</td><td class="v">${{p.type}} (${{p.status}})</td></tr>
                <tr><td class="k">${{I18N.land_district_label}}</td><td class="v">${{p.land_distr}}</td></tr>
                <tr><td class="k">${{I18N.issue_date_label}}</td><td class="v">${{p.issue_date}}</td></tr>
                <tr><td class="k">${{I18N.ownership_group_label}}</td><td class="v">Group: ${{p.group_id || '-'}} | Label: ${{p.label_group || '-'}}${{p.agg_group ? ' | Agg: ' + p.agg_group : ''}}</td></tr>
                <tr><td class="k">${{I18N.rep_block_label}}</td><td class="v">${{p.is_rep === 1 ? I18N.rep_yes : I18N.rep_no}}</td></tr>
                <tr><td class="k">${{I18N.estate_des_label}}</td><td class="v" style="font-size:11px;">${{p.estate_des}}</td></tr>
            </table>
        </div>
    `;
}}

function isOwnerCountInRange(mlcOwners, minVal, maxVal, includeUnrecorded) {{
    if (mlcOwners === null) {{
        return minVal === 0 && includeUnrecorded;
    }}
    return mlcOwners >= minVal && mlcOwners <= maxVal;
}}

function passesCurrentFilter(p) {{
    const showStd = document.getElementById('toggle-std-maori').checked;
    const showAgg = document.getElementById('toggle-agg-maori').checked;
    const catFilter = document.getElementById('ownership-filter').value;
    const filterParcelsByRange = document.getElementById('apply-range-to-parcels').checked;
    const includeUnrecorded = document.getElementById('include-unrecorded-owners').checked;
    const {{ minVal, maxVal }} = getOwnerRangeValues();

    if (p.is_aggregated === 0 && !showStd) return false;
    if (p.is_aggregated === 1 && !showAgg) return false;
    if (catFilter === 'standard' && p.is_aggregated !== 0) return false;
    if (catFilter === 'aggregated' && p.is_aggregated !== 1) return false;

    if (filterParcelsByRange) {{
        if (!isOwnerCountInRange(p.mlc_owners, minVal, maxVal, includeUnrecorded)) return false;
    }}
    return true;
}}

function applyFilters() {{
    if (maoriGeoJsonLayer) {{
        map.removeLayer(maoriGeoJsonLayer);
    }}

    let totalParcels = 0;
    let stdParcels = 0;
    let aggParcels = 0;
    let totalAreaHa = 0;

    maoriGeoJsonLayer = L.geoJSON(MAORI_GEOJSON, {{
        pane: 'maoriPane',
        renderer: sharedCanvas,
        filter: (feature) => {{
            const ok = passesCurrentFilter(feature.properties);
            if (ok) {{
                totalParcels++;
                totalAreaHa += feature.properties.area_ha;
                if (feature.properties.is_aggregated === 1) aggParcels++;
                else stdParcels++;
            }}
            return ok;
        }},
        style: getMaoriParcelStyle,
        onEachFeature: (feature, layer) => {{
            const p = feature.properties;
            titleLookup[p.title_no] = layer;
            const ownersText = p.mlc_owners !== null ? p.mlc_owners.toLocaleString() : I18N.not_recorded;
            layer.bindTooltip(
                `<strong>${{I18N.tt_title}}: ${{p.title_no}}</strong><br/>${{I18N.tt_mlc_owners}}: <b>${{ownersText}}</b><br/>${{I18N.area_label}}: ${{p.area_ha.toLocaleString()}} ha<br/><em>${{p.is_aggregated ? I18N.tt_agg : I18N.tt_std}}</em>`,
                {{ sticky: true }}
            );
            layer.bindPopup(buildMaoriPopupHtml(p), {{ maxWidth: 345 }});
            layer.on('mouseover', function() {{
                this.setStyle({{ weight: 2.5, color: '#0f172a', fillOpacity: 0.85 }});
            }});
            layer.on('mouseout', function() {{
                maoriGeoJsonLayer.resetStyle(this);
            }});
        }}
    }}).addTo(map);

    document.getElementById('stat-total-parcels').textContent = totalParcels.toLocaleString();
    document.getElementById('stat-total-area').textContent = Math.round(totalAreaHa).toLocaleString();
    document.getElementById('stat-std-parcels').textContent = stdParcels.toLocaleString();
    document.getElementById('stat-agg-parcels').textContent = aggParcels.toLocaleString();

    refreshOwnerLabels();
}}

function refreshOwnerLabels() {{
    ownerLabelsLayerGroup.clearLayers();
    const showLabels = document.getElementById('toggle-owner-labels').checked;
    const countBadge = document.getElementById('visible-labels-count');
    if (!showLabels) {{
        countBadge.textContent = I18N.status_hidden;
        return;
    }}

    const {{ minVal, maxVal }} = getOwnerRangeValues();
    const bounds = map.getBounds();
    let labelCount = 0;

    MAORI_GEOJSON.features.forEach(feat => {{
        const p = feat.properties;
        if (p.is_rep !== 1 || p.mlc_owners === null) return;
        if (p.mlc_owners < minVal || p.mlc_owners > maxVal) return;
        if (!passesCurrentFilter(p)) return;
        if (!bounds.contains([p.rep_lat, p.rep_lon])) return;

        labelCount++;
        const icon = L.divIcon({{
            className: 'owner-label-icon',
            html: `<span class="owner-label-text ${{p.is_aggregated ? 'agg' : ''}}">${{p.mlc_owners}}</span>`,
            iconSize: [0, 0]
        }});
        L.marker([p.rep_lat, p.rep_lon], {{ icon, interactive: false }}).addTo(ownerLabelsLayerGroup);
    }});

    countBadge.textContent = `${{labelCount.toLocaleString()}} ${{I18N.labels_in_view_suffix}}`;
}}

map.on('moveend', refreshOwnerLabels);

function updateMaoriOpacity(val) {{
    document.getElementById('maori-opacity-val').textContent = val + '%';
    currentMaoriOpacity = parseFloat(val) / 100.0;
    if (maoriGeoJsonLayer) {{
        maoriGeoJsonLayer.setStyle(getMaoriParcelStyle);
    }}
}}

// ============================================================================
// 3. GEOTHERMAL POWER STATIONS LAYER
// ============================================================================
const plantsLayerGroup = L.layerGroup().addTo(map);
const plantMarkers = [];

PLANTS_DATA.forEach(plant => {{
    const icon = L.divIcon({{
        className: '',
        html: `<div style="
            width: 0;
            height: 0;
            border-left: 9px solid transparent;
            border-right: 9px solid transparent;
            border-bottom: 16px solid ${{APP_CONFIG.plantMarkerColor}};
            filter: drop-shadow(0 1px 3px rgba(0,0,0,0.45));
        "></div>`,
        iconSize: [18, 16],
        iconAnchor: [9, 12]
    }});

    const marker = L.marker([plant.lat, plant.lon], {{ icon, pane: 'plantPane' }});
    marker.bindTooltip(`${{plant.name}} (${{plant.capacity_mw}} MW)`, {{
        permanent: true,
        direction: 'right',
        offset: [10, -2],
        className: 'plant-label-tooltip'
    }});
    marker.bindPopup(`
        <div class="popup-card">
            <h3 style="border-bottom-color:#dc2626;">⚡ ${{plant.name}} ${{I18N.plant_popup_suffix}}</h3>
            <table class="popup-table">
                <tr><td class="k">${{I18N.plant_cap_label}}</td><td class="v"><strong>${{plant.capacity_mw}} MW</strong></td></tr>
                <tr><td class="k">${{I18N.plant_tech_label}}</td><td class="v">${{plant.type}}</td></tr>
                <tr><td class="k">${{I18N.plant_op_label}}</td><td class="v">${{plant.operator}}</td></tr>
                <tr><td class="k">${{I18N.plant_nztm_label}}</td><td class="v">${{plant.easting}} E, ${{plant.northing}} N</td></tr>
                <tr><td class="k">${{I18N.plant_wgs_label}}</td><td class="v">${{plant.lat.toFixed(5)}}, ${{plant.lon.toFixed(5)}}</td></tr>
            </table>
        </div>
    `);
    marker.addTo(plantsLayerGroup);
    plantMarkers.push({{ marker, data: plant }});
}});

function toggleLayerGroup(name, checked) {{
    if (name === 'plants') {{
        if (checked) plantsLayerGroup.addTo(map);
        else map.removeLayer(plantsLayerGroup);
    }}
}}

function togglePlantLabels(show) {{
    plantMarkers.forEach(pm => {{
        if (show) pm.marker.openTooltip();
        else pm.marker.closeTooltip();
    }});
}}

// ============================================================================
// 4. SEARCH, BOOKMARKS & FLOATING LEGEND
// ============================================================================
function flyToBookmark(lat, lon, zoom, btnEl) {{
    document.querySelectorAll('.bookmark-grid .btn-chip').forEach(b => b.classList.remove('active'));
    if (btnEl) btnEl.classList.add('active');
    map.flyTo([lat, lon], zoom, {{ duration: 0.9 }});
}}

function handleSearch(query) {{
    const resultsBox = document.getElementById('search-results');
    const q = query.trim().toLowerCase();
    if (q.length < 2) {{
        resultsBox.style.display = 'none';
        resultsBox.innerHTML = '';
        return;
    }}

    const matches = [];
    // Search Geothermal Plants
    PLANTS_DATA.forEach(pl => {{
        if (pl.name.toLowerCase().includes(q) || pl.operator.toLowerCase().includes(q)) {{
            matches.push({{
                label: `⚡ ${{I18N.search_plant_prefix}}: <strong>${{pl.name}}</strong> (${{pl.capacity_mw}} MW — ${{pl.operator}})`,
                action: () => {{
                    map.flyTo([pl.lat, pl.lon], 13, {{ duration: 0.8 }});
                    const found = plantMarkers.find(m => m.data.name === pl.name);
                    if (found) found.marker.openPopup();
                }}
            }});
        }}
    }});

    // Search Māori Land Titles
    for (let i = 0; i < MAORI_GEOJSON.features.length && matches.length < 25; i++) {{
        const p = MAORI_GEOJSON.features[i].properties;
        if (
            p.title_no.toLowerCase().includes(q) ||
            p.estate_des.toLowerCase().includes(q) ||
            p.group_id.toLowerCase().includes(q)
        ) {{
            const ownersStr = p.mlc_owners !== null ? `${{p.mlc_owners}} ${{I18N.search_owners_suffix}}` : I18N.not_recorded;
            matches.push({{
                label: `🏞️ ${{I18N.search_title_prefix}} <strong>${{p.title_no}}</strong> (${{ownersStr}}, ${{p.area_ha}} ha) — <span style="color:#64748b;">${{p.estate_des.slice(0, 55)}}...</span>`,
                action: () => {{
                    map.flyTo([p.rep_lat, p.rep_lon], 14, {{ duration: 0.8 }});
                    const lyr = titleLookup[p.title_no];
                    if (lyr) lyr.openPopup();
                }}
            }});
        }}
    }}

    if (matches.length === 0) {{
        resultsBox.innerHTML = `<div class="search-item" style="color:#64748b;">${{I18N.no_matches}}</div>`;
        resultsBox.style.display = 'block';
        return;
    }}

    resultsBox.innerHTML = '';
    matches.forEach(m => {{
        const div = document.createElement('div');
        div.className = 'search-item';
        div.innerHTML = m.label;
        div.onclick = () => {{
            m.action();
            resultsBox.style.display = 'none';
        }};
        resultsBox.appendChild(div);
    }});
    resultsBox.style.display = 'block';
}}

function toggleSidebar() {{
    const sb = document.getElementById('sidebar');
    const btn = document.getElementById('sidebar-toggle');
    sb.classList.toggle('collapsed');
    btn.textContent = sb.classList.contains('collapsed') ? I18N.show_panel : I18N.hide_panel;
    setTimeout(() => map.invalidateSize(), 280);
}}

// Floating Legend (Bottom Left)
const legendControl = L.control({{ position: 'bottomleft' }});
legendControl.onAdd = function() {{
    const div = L.DomUtil.create('div', 'map-legend');
    div.innerHTML = `
        <h4>${{I18N.legend_title}}</h4>
        <div class="legend-row">
            <span class="swatch" style="background:{STYLE_STANDARD_MAORI["fillColor"]}; border:1.5px solid {STYLE_STANDARD_MAORI["color"]};"></span>
            <span>${{I18N.leg_std_maori}}</span>
        </div>
        <div class="legend-row">
            <span class="swatch" style="background:{STYLE_AGGREGATED_MAORI["fillColor"]}; border:1.5px solid {STYLE_AGGREGATED_MAORI["color"]};"></span>
            <span>${{I18N.leg_agg_maori}}</span>
        </div>
        <div class="legend-row">
            <span class="swatch" style="background:#f43f5e33; border:2px dashed #be123c;"></span>
            <span>${{I18N.leg_tahu}}</span>
        </div>
        <div class="legend-row">
            <span class="swatch" style="background:#0ea5e933; border:2px dashed #0369a1;"></span>
            <span>${{I18N.leg_tuwharetoa}}</span>
        </div>
        <div class="legend-row">
            <span class="swatch-triangle"></span>
            <span>${{I18N.leg_plant}}</span>
        </div>
        <div class="legend-note">
            ${{I18N.legend_note}}
        </div>
    `;
    return div;
}};
legendControl.addTo(map);

// Cursor Coordinate Readout (Bottom Right)
const coordControl = L.control({{ position: 'bottomright' }});
coordControl.onAdd = function() {{
    const div = L.DomUtil.create('div', 'coord-box');
    div.id = 'coord-readout';
    div.textContent = 'Lat: -38.5500, Lon: 176.1800';
    return div;
}};
coordControl.addTo(map);

map.on('mousemove', (e) => {{
    document.getElementById('coord-readout').textContent =
        `Lat: ${{e.latlng.lat.toFixed(4)}}, Lon: ${{e.latlng.lng.toFixed(4)}}`;
}});

// Initial render
applyFilters();
</script>
</body>
</html>"""

# Write English Map App (index.html)
html_en = build_html_app("en")
with open(OUTPUT_HTML_EN_PATH, "w", encoding="utf-8") as f:
    f.write(html_en)

# Write Japanese Map App (index_ja.html)
html_ja = build_html_app("ja")
with open(OUTPUT_HTML_JA_PATH, "w", encoding="utf-8") as f:
    f.write(html_ja)

en_size_mb = OUTPUT_HTML_EN_PATH.stat().st_size / (1024 * 1024)
ja_size_mb = OUTPUT_HTML_JA_PATH.stat().st_size / (1024 * 1024)
print(f"Generated English Map App:  {OUTPUT_HTML_EN_PATH.resolve()} ({en_size_mb:.2f} MB)")
print(f"Generated Japanese Map App: {OUTPUT_HTML_JA_PATH.resolve()} ({ja_size_mb:.2f} MB)")


Generated English Map App:  C:\★大学院留学に向けて\01_On-site\11. Geothermal Research Project\★NEDO Project Phase 2\data\map_app\index.html (5.54 MB)
Generated Japanese Map App: C:\★大学院留学に向けて\01_On-site\11. Geothermal Research Project\★NEDO Project Phase 2\data\map_app\index_ja.html (5.54 MB)
